ReAct (Reasoning + Acting) agent from scratch using LangGraph. The agent receives a question, decides whether it needs a tool, calls the tool, observes the result, and loops back to reason again — until it has enough information to give a final answer.

Imports and Setup

In [1]:
from typing import Literal
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode

C:\Users\SANDEEP S\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0,             # temperature=0 for consistent, deterministic reasoning
    base_url="http://localhost:11434",
)

response = llm.invoke([HumanMessage(content="Reply with one word: Ready")])
print("Ollama connected:", response.content)

Ollama connected: Ready


Defining Tools:

In [3]:
# ============================================================
# SECTION 1 - Define Tools
# Ref: https://docs.langchain.com/oss/python/langchain/tools
#
# Tools are plain Python functions decorated with @tool.
# The decorator does three things:
#   - Registers the function name as the tool name
#   - Uses the docstring as the tool description the LLM reads
#     to decide whether to call this tool
#   - Builds the input schema from the function signature
# ============================================================

# -- 1.1 CALCULATOR TOOL -------------------------------------
# The LLM will call this when the question involves arithmetic.
# Without this tool, the LLM would guess the math result.
# With it, the agent delegates computation to actual Python.

@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result.
    Use this tool whenever the user asks for any arithmetic,
    such as addition, subtraction, multiplication, or division.
    Input must be a valid Python math expression, e.g. '12 * 8 + 5'.
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"


# -- 1.2 WORD COUNTER TOOL -----------------------------------
# The LLM will call this when the question involves counting
# words in a given piece of text.

@tool
def word_counter(text: str) -> str:
    """
    Counts the number of words in a given text and returns the count.
    Use this tool when the user asks how many words are in a sentence
    or paragraph.
    Input must be the text whose words you want to count.
    """
    count = len(text.split())
    return f"The text contains {count} words."


# -- 1.3 STRING REVERSAL TOOL --------------------------------
# The LLM will call this when the question asks to reverse a string.

@tool
def string_reverser(text: str) -> str:
    """
    Reverses the characters in a given string and returns the result.
    Use this tool when the user asks to reverse a word, sentence,
    or any piece of text.
    Input must be the string to reverse.
    """
    return text[::-1]


# -- 1.4 REGISTER TOOLS --------------------------------------
# Collect all tools into a list. This list is used in two places:
#   - bind_tools: tells the LLM what tools are available
#   - ToolNode: actually executes whichever tool the LLM calls

tools = [calculator, word_counter, string_reverser]

# Bind tools to the LLM so it knows their names, descriptions,
# and input schemas when deciding what action to take.
llm_with_tools = llm.bind_tools(tools)

# -- 1.5 VERIFY TOOLS ----------------------------------------
print("Registered tools:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

Registered tools:
  - calculator: Evaluates a mathematical expression and returns the result.
...
  - word_counter: Counts the number of words in a given text and returns the c...
  - string_reverser: Reverses the characters in a given string and returns the re...


Building ReAct graph:

In [4]:
def agent_node(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}



tool_node = ToolNode(tools)



def should_continue(state: MessagesState) -> Literal["tools", END]:
    last_message = state["messages"][-1]

    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

    return END


builder = StateGraph(MessagesState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        END: END,
    }
)

# After tools executes, always go back to agent to reason
# about the tool result before deciding what to do next.
builder.add_edge("tools", "agent")

react_graph = builder.compile()

print("ReAct graph compiled successfully.")
print()
print("Graph structure:")
print("  START -> agent")
print("  agent -> should_continue")
print("         -> tools (if tool call requested)")
print("         -> END   (if final answer ready)")
print("  tools -> agent")

ReAct graph compiled successfully.

Graph structure:
  START -> agent
  agent -> should_continue
         -> tools (if tool call requested)
         -> END   (if final answer ready)
  tools -> agent


In [ ]:
User asks a question
  Agent reads it, requests calculator tool
    Calculator runs, result added to history
  Agent reads history + result, gives final answer
Exit

In [5]:
# walks through the message history and prints each step
# in a readable format so you can see the full
# Reason -> Act -> Observe loop clearly.
def run_agent(question: str):
    print("=" * 60)
    print("Question:", question)
    print("-" * 60)

    result = react_graph.invoke({
        "messages": [HumanMessage(content=question)]
    })

    for message in result["messages"]:
        if isinstance(message, HumanMessage):
            print("User        :", message.content)

        elif isinstance(message, AIMessage):
            if message.tool_calls:
                for tc in message.tool_calls:
                    print("Agent       : requesting tool ->", tc["name"])
                    print("  args      :", tc["args"])
            else:
                print("Agent       :", message.content)

        elif isinstance(message, ToolMessage):
            print("Tool result :", message.content)

    print("=" * 60)
    print()


Testing the Agent:

In [6]:
run_agent("What is 347 multiplied by 28?")

Question: What is 347 multiplied by 28?
------------------------------------------------------------
User        : What is 347 multiplied by 28?
Agent       : requesting tool -> calculator
  args      : {'expression': '347 * 28'}
Tool result : 9716
Agent       : 347 multiplied by 28 is 9716.



In [7]:
run_agent("How many words are in this sentence: The quick brown fox jumps over the lazy dog")


Question: How many words are in this sentence: The quick brown fox jumps over the lazy dog
------------------------------------------------------------
User        : How many words are in this sentence: The quick brown fox jumps over the lazy dog
Agent       : requesting tool -> word_counter
  args      : {'text': 'The quick brown fox jumps over the lazy dog'}
Tool result : The text contains 9 words.
Agent       : The sentence you provided contains 9 words.



In [8]:
run_agent("Can you reverse the word: LangGraph")

Question: Can you reverse the word: LangGraph
------------------------------------------------------------
User        : Can you reverse the word: LangGraph
Agent       : requesting tool -> string_reverser
  args      : {'text': 'LangGraph'}
Tool result : hparGgnaL
Agent       : The reversed word of "LangGraph" is "hparGgnaL".



In [9]:
run_agent("What is the capital of France?")

Question: What is the capital of France?
------------------------------------------------------------
User        : What is the capital of France?
Agent       : The capital of France is Paris. If you need any further assistance related to this or any other information, feel free to ask!

